In [1]:
import itertools
import logging
import os

import mlflow

from model.train_models import train_evaluate_model
from utils.data_prep import get_clean_combined_data

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

mlflow.sklearn.autolog(disable=True)

DOWNLOAD = False  # Set to use local data for testing

In [2]:
completed_runs_file = "completed_runs.txt"

if os.path.exists(completed_runs_file):
    with open(completed_runs_file, "r") as f:
        completed_runs = {line.strip() for line in f if line.strip()}
else:
    completed_runs = set()

print(f"Loaded {len(completed_runs)} completed runs from memory.")

Loaded 19 completed runs from memory.


In [3]:
ks = [0.25, 0.5, 0.75, 1]
ns = [4, 5]
event_cols = ["sub_event_type", "event_type"]
remove_abyei_options = [False]  # TODO Abyei currently not present for rainfall
include_food_options = [True, False]
include_rain_options = [True, False]
include_text_options = [True, False]

In [4]:
xgb_params = {
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "max_delta_step": [0, 1, 5],
    "gamma": [0, 1, 3, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1, 2],
    "reg_lambda": [1, 5, 10],
    "colsample_bylevel": [0.6, 0.8, 1.0],
}

In [ ]:
data_configs = itertools.product(
    remove_abyei_options,
    include_food_options,
    include_rain_options,
    include_text_options,
    ks,
    event_cols,
)

for (
    remove_abyei,
    include_food,
    include_rain,
    include_text,
    k,
    event_col,
) in data_configs:
    data_sources = [
        src
        for src, include in zip(
            ["food", "rain", "text"], [include_food, include_rain, include_text]
        )
        if include
    ]

    food_str = "_food" if include_food else ""
    rain_str = "_rain" if include_rain else ""
    text_str = "_text" if include_text else ""
    abyei_str = "_remove_abyei" if remove_abyei else ""
    event_str = "event" if event_col == "event_type" else "sub"

    which_data = f"acled_{event_str}{food_str}{rain_str}{text_str}{abyei_str}"
    model_data, predictor_cols = get_clean_combined_data(
        data_sources=data_sources,
        download=DOWNLOAD,
        remove_abyei=remove_abyei,
        k=k,
        event_col=event_col,
    )
    if include_text:
        pca_options = [True, False]  # Run both when text is included
    else:
        pca_options = [False]  # Only run without PCA when text isn't included

    for n in ns:
        for use_pca in pca_options:
            pca_str = "_pca" if use_pca else ""
            run_name = f"{which_data}{pca_str}_{k}_{n}"

            if run_name in completed_runs:
                print(f"Skipping already completed run: {run_name}")
                continue

            all_params = {
                **xgb_params,
                "k": k,
                "event_col": event_col,
                "remove_abyei": remove_abyei,
                "n_splits": n,
                "use_pca": use_pca,
            }

            with mlflow.start_run(run_name=run_name):
                mlflow.set_tags(
                    {
                        "data_version": which_data,
                        "remove_abyei": remove_abyei,
                        "include_food": include_food,
                        "include_rain": include_rain,
                        "include_text": include_text,
                        "use_pca": use_pca,
                        "k": k,
                        "n_splits": n,
                        "event_col": event_col,
                    }
                )
                logger.info(f"Running mode: {run_name}")

                results, best_params = train_evaluate_model(
                    model_data,
                    predictor_cols,
                    all_params,
                    best_params=False,
                    use_pca=use_pca,
                )

                mlflow.log_params(best_params)
                mlflow.log_metrics({key: float(val) for key, val in results.items()})
                mlflow.log_dict(results, "model_report.json")

                completed_runs.add(run_name)  # Add to log file
                with open(completed_runs_file, "a") as f:
                    f.write(run_name + "\n")

INFO:ACLED processing:Data grouped by sub_event_type
INFO:ACLED processing:Escalation target set at 0.25 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:hdx.api.configuration:No HDX base configuration parameter. Using default base configuration file: C:\Users\evely\PycharmProjects\msc-final-project\.venv\Lib\site-packages\hdx\api\hdx_base_configuration.yaml.
INFO:hdx.api.configuration:Loading HDX base configuration from: C:\Users\evely\PycharmProjects\msc-final-project\.venv\Lib\site-packages\hdx\api\hdx_base_configuration.yaml
INFO:hdx.api.configuration:No HDX configuration parameter and no configuration file at default path: C:\Users\evely\.hdx_configuration.yaml.
INFO:hdx.api.configuration:Read only access to HDX: True
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\Sudan - Food Prices.csv
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\South Sudan - Food Prices.csv
INFO:Data preparation:Food prices data processed.
I

Skipping already completed run: acled_sub_food_rain_text_pca_0.25_4
Skipping already completed run: acled_sub_food_rain_text_0.25_4
Skipping already completed run: acled_sub_food_rain_text_pca_0.25_5
Skipping already completed run: acled_sub_food_rain_text_0.25_5


INFO:ACLED processing:Data grouped by event_type
INFO:ACLED processing:Escalation target set at 0.25 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\Sudan - Food Prices.csv
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\South Sudan - Food Prices.csv
INFO:Data preparation:Food prices data processed.
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\sdn-rainfall-subnat-full.csv
INFO:Data preparation:Rainfall data processed.
INFO:Data preparation:Notes data processed.


Skipping already completed run: acled_event_food_rain_text_pca_0.25_4
Skipping already completed run: acled_event_food_rain_text_0.25_4
Skipping already completed run: acled_event_food_rain_text_pca_0.25_5
Skipping already completed run: acled_event_food_rain_text_0.25_5


INFO:ACLED processing:Data grouped by sub_event_type
INFO:ACLED processing:Escalation target set at 0.5 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\Sudan - Food Prices.csv
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\South Sudan - Food Prices.csv
INFO:Data preparation:Food prices data processed.
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\sdn-rainfall-subnat-full.csv
INFO:Data preparation:Rainfall data processed.
INFO:Data preparation:Notes data processed.


...

Skipping already completed run: acled_event_food_rain_text_pca_0.5_4
Skipping already completed run: acled_event_food_rain_text_0.5_4
Skipping already completed run: acled_event_food_rain_text_pca_0.5_5
Skipping already completed run: acled_event_food_rain_text_0.5_5


INFO:ACLED processing:Data grouped by sub_event_type
INFO:ACLED processing:Escalation target set at 0.75 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\Sudan - Food Prices.csv
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\South Sudan - Food Prices.csv
INFO:Data preparation:Food prices data processed.
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\sdn-rainfall-subnat-full.csv
INFO:Data preparation:Rainfall data processed.
INFO:Data preparation:Notes data processed.


Skipping already completed run: acled_sub_food_rain_text_pca_0.75_4
Skipping already completed run: acled_sub_food_rain_text_0.75_4
Skipping already completed run: acled_sub_food_rain_text_pca_0.75_5


INFO:__main__:Running mode: acled_sub_food_rain_text_0.75_5
INFO:Cross validation:Cross-validation testing splits:


--- Fold 1 ---
Train window: 2018-01 to 2018-10 (180 rows)
Test window:  2018-11 to 2019-08 (180 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-08 (360 rows)
Test window:  2019-09 to 2020-06 (180 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-06 (540 rows)
Test window:  2020-07 to 2021-04 (180 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-04 (720 rows)
Test window:  2021-05 to 2022-02 (180 rows)
------------------------------
--- Fold 5 ---
Train window: 2018-01 to 2022-02 (900 rows)
Test window:  2022-03 to 2022-12 (180 rows)
------------------------------
